# SVM 多因子量化选股策略

## 论文参考

- **标题**: Research on Multi-factor Quantitative Investment Strategy of SVM Model
- **作者**: Ting Liu
- **年份**: 2023
- **DOI**: 10.54254/2754-1169/49/20230537

### 摘要

本文研究了基于支持向量机 (SVM) 的多因子量化投资策略。采用 RBF 核函数的 SVM 模型
对 A 股进行二分类选股（上涨/下跌），通过 GridSearchCV 优化超参数 C 和 gamma。
由于 SVM 对特征尺度敏感，所有因子经过 StandardScaler 标准化处理。
策略在回测期内取得了 **累计收益率 41.25%**，展示了 SVM 在金融预测中的潜力。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### SVM (Support Vector Machine) 分类

**1. 线性 SVM 原始问题**

$$\min_{w, b} \frac{1}{2}\|w\|^2 + C\sum_{i=1}^{n} \xi_i$$
$$\text{s.t.}\quad y_i(w^T x_i + b) \geq 1 - \xi_i, \quad \xi_i \geq 0$$

其中 $C$ 是惩罚参数，$\xi_i$ 是松弛变量。

**2. 对偶问题**

$$\max_{\alpha} \sum_{i=1}^{n} \alpha_i - \frac{1}{2}\sum_{i,j} \alpha_i \alpha_j y_i y_j K(x_i, x_j)$$
$$\text{s.t.}\quad 0 \leq \alpha_i \leq C, \quad \sum_{i=1}^{n}\alpha_i y_i = 0$$

**3. RBF (径向基) 核函数**

$$K(x_i, x_j) = \exp\left(-\gamma \|x_i - x_j\|^2\right)$$

其中 $\gamma = \frac{1}{2\sigma^2}$ 控制决策边界的复杂度：
- $\gamma$ 大 $\Rightarrow$ 每个支持向量影响范围小，边界复杂，可能过拟合
- $\gamma$ 小 $\Rightarrow$ 每个支持向量影响范围大，边界平滑，可能欠拟合

**4. 决策函数**

$$f(x) = \text{sign}\left(\sum_{i \in SV} \alpha_i y_i K(x_i, x) + b\right)$$

**5. 标准化的必要性**

RBF 核基于欧氏距离，特征尺度不同会导致距离失真：
$$x' = \frac{x - \mu}{\sigma} \quad (\text{Z-score 标准化})$$

**6. GridSearchCV 超参数优化**

$$\text{Best}(C^*, \gamma^*) = \arg\max_{C, \gamma} \text{CV-Accuracy}(C, \gamma)$$

In [ ]:
# ============================================================
# Cell 3: 动画展示 - SVM 决策边界 (2D PCA 投影)
# ============================================================
import numpy as np
import plotly.graph_objects as go
from sklearn.svm import SVC
from sklearn.datasets import make_circles
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X_demo, y_demo = make_circles(n_samples=200, noise=0.1, factor=0.4, random_state=42)
scaler_demo = StandardScaler()
X_demo = scaler_demo.fit_transform(X_demo)

xx, yy = np.meshgrid(np.linspace(-3, 3, 100), np.linspace(-3, 3, 100))
grid = np.column_stack([xx.ravel(), yy.ravel()])

# 不同 C 和 gamma 的组合
configs = [
    {'C': 0.1, 'gamma': 0.1},
    {'C': 1.0, 'gamma': 0.1},
    {'C': 1.0, 'gamma': 1.0},
    {'C': 10.0, 'gamma': 1.0},
    {'C': 10.0, 'gamma': 5.0},
    {'C': 100.0, 'gamma': 10.0},
]

frames = []
for cfg in configs:
    svc = SVC(kernel='rbf', C=cfg['C'], gamma=cfg['gamma'], probability=True, random_state=42)
    svc.fit(X_demo, y_demo)
    prob_grid = svc.predict_proba(grid)[:, 1].reshape(xx.shape)
    acc = svc.score(X_demo, y_demo)
    n_sv = svc.n_support_.sum()

    frames.append(go.Frame(
        data=[
            go.Contour(x=np.linspace(-3, 3, 100), y=np.linspace(-3, 3, 100),
                       z=prob_grid, colorscale='RdBu_r', opacity=0.6,
                       contours=dict(showlines=True),
                       showscale=True, colorbar=dict(title='P(class=1)')),
            go.Scatter(x=X_demo[y_demo == 0, 0], y=X_demo[y_demo == 0, 1],
                       mode='markers', name='Class 0',
                       marker=dict(color='#2196F3', size=5, line=dict(width=0.5, color='black'))),
            go.Scatter(x=X_demo[y_demo == 1, 0], y=X_demo[y_demo == 1, 1],
                       mode='markers', name='Class 1',
                       marker=dict(color='#F44336', size=5, line=dict(width=0.5, color='black'))),
            # 支持向量
            go.Scatter(x=X_demo[svc.support_, 0], y=X_demo[svc.support_, 1],
                       mode='markers', name=f'SV ({n_sv})',
                       marker=dict(color='rgba(0,0,0,0)', size=12,
                                   line=dict(width=2, color='gold'))),
        ],
        name=f'C={cfg["C"]}, gamma={cfg["gamma"]} (acc={acc:.2%}, SV={n_sv})'
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title=dict(text='SVM RBF核: C 和 gamma 对决策边界的影响'),
        xaxis_title='Feature 1 (标准化)', yaxis_title='Feature 2 (标准化)',
        height=600, width=700,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='▶ 播放', method='animate',
                 args=[None, {'frame': {'duration': 1000}, 'transition': {'duration': 500}}]),
            dict(label='⏸ 暂停', method='animate',
                 args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
            active=0,
        )],
    )
)
fig.show()

In [ ]:
# ============================================================
# Cell 4: 环境设置与导入
# ============================================================
import sys
import os
import warnings
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA
from datetime import datetime

warnings.filterwarnings('ignore')
np.random.seed(42)

sys.path.insert(0, '..')
from shared.data_fetcher import get_stock_daily, get_index_daily, get_multiple_stocks_daily
from shared.backtest_engine import MultiStockBacktester
from shared.factors import (
    momentum, volatility, rsi, macd, bollinger_bands, atr,
    turnover_factor, volume_price_corr, high_low_range, price_to_ma,
    winsorize, standardize
)
from shared.visualization import (
    plot_equity_curve, plot_drawdown, plot_metrics_table,
    plot_factor_importance, plot_returns_distribution
)
from shared.constants import DEFAULT_START, DEFAULT_END, CSI300_CODE

print('所有模块导入成功')
print(f'回测区间: {DEFAULT_START} - {DEFAULT_END}')

In [ ]:
# ============================================================
# Cell 5: 数据获取
# ============================================================

STOCK_POOL = [
    '600519', '601318', '600036', '000858', '601166',
    '600276', '601398', '600030', '000333', '002415',
    '600900', '601888', '000568', '002304', '601012',
    '600031', '603259', '600585', '000661', '601668',
]

try:
    stock_data = get_multiple_stocks_daily(STOCK_POOL, DEFAULT_START, DEFAULT_END)
    print(f'成功获取 {len(stock_data)} 只股票数据')
except Exception as e:
    print(f'数据获取异常: {e}, 使用模拟数据')
    dates = pd.bdate_range(DEFAULT_START, DEFAULT_END)
    stock_data = {}
    for sym in STOCK_POOL:
        np.random.seed(hash(sym) % 2**31)
        price = 50 + np.cumsum(np.random.randn(len(dates)) * 0.5)
        price = np.maximum(price, 5)
        stock_data[sym] = pd.DataFrame({
            'open': price * (1 + np.random.randn(len(dates)) * 0.005),
            'close': price,
            'high': price * (1 + np.abs(np.random.randn(len(dates)) * 0.01)),
            'low': price * (1 - np.abs(np.random.randn(len(dates)) * 0.01)),
            'volume': np.random.randint(100000, 5000000, len(dates)).astype(float),
            'turnover': np.random.uniform(0.3, 5.0, len(dates)),
        }, index=dates)

try:
    benchmark = get_index_daily(CSI300_CODE, DEFAULT_START, DEFAULT_END)
    benchmark_prices = benchmark['close']
    print(f'基准数据: {len(benchmark_prices)} 条')
except Exception as e:
    print(f'基准获取异常: {e}')
    dates = pd.bdate_range(DEFAULT_START, DEFAULT_END)
    benchmark_prices = pd.Series(5000 + np.cumsum(np.random.randn(len(dates)) * 10), index=dates)

In [ ]:
# ============================================================
# Cell 6: 因子工程 (必须标准化!)
# ============================================================

def build_features_svm(df):
    """为单只股票构建因子"""
    features = pd.DataFrame(index=df.index)

    # 动量
    mom = momentum(df['close'], periods=[5, 10, 20])
    features = pd.concat([features, mom], axis=1)

    # 波动率
    vol = volatility(df['close'], windows=[10, 20])
    features = pd.concat([features, vol], axis=1)

    # RSI, MACD, Bollinger
    features['rsi_14'] = rsi(df['close'], 14)
    macd_df = macd(df['close'])
    features['macd_hist'] = macd_df['histogram']
    bb = bollinger_bands(df['close'], 20)
    features['bb_pctb'] = bb['bb_pctb']
    features['bb_width'] = bb['bb_width']

    # 换手率
    if 'turnover' in df.columns:
        features['turnover'] = df['turnover']
        features['turnover_ma5'] = df['turnover'].rolling(5).mean()

    # 量价相关
    if 'volume' in df.columns:
        features['vp_corr_20'] = volume_price_corr(df['close'], df['volume'], 20)

    # 价格偏离
    ptm = price_to_ma(df['close'], windows=[5, 10, 20])
    features = pd.concat([features, ptm], axis=1)

    # ATR
    if all(c in df.columns for c in ['high', 'low']):
        features['atr_14'] = atr(df['high'], df['low'], df['close'], 14)
        features['hl_range'] = high_low_range(df['high'], df['low'], df['close'])

    return features


all_features = []
for sym, df in stock_data.items():
    if len(df) < 60:
        continue
    feats = build_features_svm(df)
    feats['fwd_return_20d'] = df['close'].pct_change(20).shift(-20)
    feats['symbol'] = sym
    all_features.append(feats)

panel = pd.concat(all_features).reset_index()
if 'index' in panel.columns:
    panel.rename(columns={'index': 'date'}, inplace=True)

FEATURE_COLS = [c for c in panel.columns if c not in ['date', 'symbol', 'fwd_return_20d', 'label']]

# 二分类标签: 中位数以上=1, 以下=0
panel['date'] = pd.to_datetime(panel['date'])
panel['year_month'] = panel['date'].dt.to_period('M')

def assign_binary_label(group):
    ret = group['fwd_return_20d']
    median = ret.median()
    group['label'] = (ret >= median).astype(int)
    return group

panel = panel.groupby('year_month', group_keys=False).apply(assign_binary_label)

print(f'因子面板: {panel.shape[0]} 行 x {len(FEATURE_COLS)} 个因子')
print(f'标签分布:\n{panel["label"].value_counts()}')

In [ ]:
# ============================================================
# Cell 7: SVM 模型训练 (GridSearchCV + 滚动窗口)
# ============================================================

TRAIN_MONTHS = 12
TOP_K = 10
# 每 6 个月做一次 GridSearch, 其余月份用上次最优参数
GRID_SEARCH_INTERVAL = 6

months = sorted(panel['year_month'].unique())
print(f'数据覆盖 {len(months)} 个月')

# GridSearch 参数空间 (简化以加速)
param_grid = {
    'C': [0.1, 1.0, 10.0],
    'gamma': [0.01, 0.1, 1.0],
}

selections = {}
best_params_history = []
cv_scores_history = []
best_C, best_gamma = 1.0, 0.1  # 默认参数

for i in range(TRAIN_MONTHS, len(months) - 1):
    train_months_range = months[i - TRAIN_MONTHS: i]
    test_month = months[i]

    train_data = panel[panel['year_month'].isin(train_months_range)].copy()
    test_data = panel[panel['year_month'] == test_month].copy()

    train_data = train_data.dropna(subset=FEATURE_COLS + ['label'])
    test_data_clean = test_data.dropna(subset=FEATURE_COLS)

    if len(train_data) < 30 or len(test_data_clean) < 5:
        continue

    X_train = train_data[FEATURE_COLS].values
    y_train = train_data['label'].values.astype(int)
    X_test = test_data_clean[FEATURE_COLS].values

    # **关键**: StandardScaler 标准化 (SVM 必需!)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # 定期 GridSearchCV
    idx = i - TRAIN_MONTHS
    if idx % GRID_SEARCH_INTERVAL == 0:
        grid_search = GridSearchCV(
            SVC(kernel='rbf', probability=True, random_state=42),
            param_grid,
            cv=3,
            scoring='accuracy',
            n_jobs=-1,
        )
        grid_search.fit(X_train, y_train)
        best_C = grid_search.best_params_['C']
        best_gamma = grid_search.best_params_['gamma']
        best_score = grid_search.best_score_
        best_params_history.append({'month': str(test_month), 'C': best_C, 'gamma': best_gamma})
        cv_scores_history.append(best_score)
        print(f'  GridSearch {test_month}: C={best_C}, gamma={best_gamma}, CV={best_score:.3f}')

    # 用最优参数训练 SVM
    svm = SVC(kernel='rbf', C=best_C, gamma=best_gamma,
              probability=True, random_state=42)
    svm.fit(X_train, y_train)

    # 预测上涨概率
    prob = svm.predict_proba(X_test)[:, 1]
    test_data_clean = test_data_clean.copy()
    test_data_clean['prob_up'] = prob

    last_day = test_data_clean.groupby('symbol')['prob_up'].last()
    top_stocks = last_day.nlargest(TOP_K).index.tolist()

    rebal_date = test_data_clean['date'].max()
    selections[rebal_date] = top_stocks

print(f'\n共生成 {len(selections)} 期选股结果')
print(f'\nGridSearch 历史最优参数:')
for p in best_params_history:
    print(f"  {p['month']}: C={p['C']}, gamma={p['gamma']}")

In [ ]:
# ============================================================
# Cell 8: 信号生成与回测
# ============================================================

all_prices = {sym: df['close'] for sym, df in stock_data.items()}

backtester = MultiStockBacktester(initial_capital=1_000_000, rebalance_freq='M')
result = backtester.run(
    all_prices=all_prices,
    selections=selections,
    benchmark_prices=benchmark_prices
)

print('=== SVM 多因子选股策略回测结果 ===')
for k, v in result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 9: 绩效可视化
# ============================================================
import matplotlib.pyplot as plt

# 1. 收益曲线
benchmark_equity = None
if result.get('benchmark_returns') is not None:
    benchmark_equity = (1 + result['benchmark_returns']).cumprod() * result['equity_curve'].iloc[0]
plot_equity_curve(result['equity_curve'], benchmark_equity,
                  title='SVM 多因子选股 - 累计收益')

# 2. 回撤图
plot_drawdown(result['equity_curve'], title='SVM 多因子选股 - 回撤')

# 3. GridSearch CV 分数变化
if cv_scores_history:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(range(len(cv_scores_history)), cv_scores_history,
           color='#2196F3', alpha=0.8)
    ax.set_title('SVM GridSearchCV 最佳准确率 (定期更新)', fontsize=14, fontweight='bold')
    ax.set_xlabel('GridSearch 周期', fontsize=12)
    ax.set_ylabel('CV 准确率', fontsize=12)
    ax.set_ylim(0.4, 0.8)
    for idx_bar, val in enumerate(cv_scores_history):
        ax.text(idx_bar, val + 0.01, f'{val:.3f}', ha='center', fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# 4. 参数变化可视化
if best_params_history:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    param_months = [p['month'] for p in best_params_history]
    c_vals = [p['C'] for p in best_params_history]
    gamma_vals = [p['gamma'] for p in best_params_history]

    axes[0].plot(range(len(c_vals)), c_vals, 'o-', color='#F44336', linewidth=2)
    axes[0].set_title('最优 C 参数变化', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('C')
    axes[0].set_yscale('log')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(range(len(gamma_vals)), gamma_vals, 's-', color='#4CAF50', linewidth=2)
    axes[1].set_title('最优 gamma 参数变化', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('gamma')
    axes[1].set_yscale('log')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# 5. 收益率分布
plot_returns_distribution(result['returns'], title='SVM 策略日收益率分布')

# 6. 绩效表
plot_metrics_table(result['metrics'], title='SVM 多因子选股策略绩效指标')

## 结果讨论

### SVM 在量化选股中的特点

1. **核技巧的优势**: RBF 核将低维线性不可分的数据映射到高维空间，
   能捕捉因子之间的非线性交互
2. **标准化至关重要**: SVM 对特征尺度极其敏感，不标准化的因子会导致
   距离计算被大尺度特征主导
3. **支持向量的稀疏性**: 决策边界仅由少量支持向量决定，具有一定的鲁棒性

### GridSearchCV 超参数优化

- C 控制误分类惩罚: C 大则模型更严格拟合训练数据
- gamma 控制决策边界复杂度: gamma 大则边界更复杂
- 定期更新超参数可以适应市场状态变化

### SVM 的局限性

| 缺点 | 说明 |
|------|------|
| 训练速度慢 | 时间复杂度 O(n^2)~O(n^3)，样本量大时不适用 |
| 无法输出特征重要性 | 不像树模型可以直接看因子贡献 |
| 概率校准问题 | SVM 的概率估计需要 Platt scaling |
| 超参数敏感 | C 和 gamma 需要仔细调优 |

### 改进方向

- 使用 PCA 降维后再训练 SVM，减少计算开销
- 尝试 LinearSVC 处理大规模数据
- 使用 Bayesian Optimization 替代 GridSearch
- 结合 SVM 与树模型做模型 Stacking